# Nonadiabatic Dynamics with a Machine-Learned Hamiltonian (hippynn)

## Table of Content: <a name="TOC"></a>

1. [Generic setups](#1)

2. [The model and its labels](#2)

3. [Choose the Nonadiabatic Dynamics Methodology](#3)

4. [Choose the initial conditions: Nuclear and Electronic](#4)

5. [Running the calculations](#5)

6. [Plotting the results](#6)

7. [The ensemble result](#7)

8. [Using your own hippynn model](#8)


## A. Learning objectives

- to drive Libra's nonadiabatic dynamics with a neural network instead of an analytical
  model Hamiltonian or an electronic structure package
- to understand what a machine-learning model has to predict for this to work, and how it
  is connected to Libra through `libra_py.packages.hippynn`
- to adapt the same interface to a hippynn model of your own

## B. Use cases

- Nonadiabatic dynamics with a machine-learned Hamiltonian
- Ehrenfest dynamics in the diabatic representation
- Vibronic beating and electronic decoherence in a molecular dimer

Analytical model Hamiltonians are cheap but idealized; electronic structure is realistic but
expensive. A trained machine-learning model sits in between: it reproduces the electronic
structure of a specific system at a small fraction of the cost, which makes ensembles of
trajectories affordable for systems where they otherwise would not be.

Here the network predicts the **diabatic electronic Hamiltonian** of a 122-atom molecular
dimer, so that everything Libra needs, including the couplings that drive transitions
between the excited states, comes from a single learned object.

## C. Functions

- `libra_py`
  - `dynamics`
    - `tsh`
      - `compute`
        - [`generic_recipe`](#generic_recipe-1)
  - `models`
    - `NQDH_heterodimer`
      - [`compute_model`](#compute_model-1) | [`also here`](#compute_model-2)
      - [`get_atomic_numbers`](#get_atomic_numbers-1)
      - [`get_initial_conditions`](#get_initial_conditions-1)
  - `packages`
    - `hippynn`
      - `methods`
        - [`hippynn_model`](#hippynn_model-1)


## D. Classes and class members


## E. Requirements

- `libra`
- [`hippynn`](https://github.com/lanl/hippynn) and `pytorch`. They are imported only when
  the network is evaluated, so the rest of `libra_py` is unaffected if they are absent.


## 1. Generic setups
<a name="1"></a>[Back to TOC](#TOC)

Two pieces of Libra are involved, and it is worth keeping them apart:

* `libra_py.packages.hippynn` is the **interface**. It knows how to evaluate any trained
  hippynn model and turn it into the Hamiltonian and derivatives that Libra expects. It is
  not tied to a molecule, in the same way that the DFTB+ or CP2K interfaces are not.
* `libra_py.models.NQDH_heterodimer` is a **trained model** distributed with Libra, which
  uses that interface. Like the LVC or Shin-Metiu models it describes one particular
  system, except that its parameters are network weights rather than a few constants.

This tutorial uses the second, which internally calls the first. Section 8 shows how to use
the interface directly with a model of your own.


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

from liblibra_core import *
from libra_py import units
import libra_py.dynamics.tsh.compute as tsh_dynamics

from libra_py.models import NQDH_heterodimer as model

## 2. The model and its labels
<a name="2"></a>[Back to TOC](#TOC)

The system is a halofluorescein AB heterodimer: two chromophores whose excited states are
delocalized combinations of excitations on either half. Its two lowest excited states, S1
and S2, are close in energy and coupled to each other, and the coupling depends sharply on
the nuclear geometry.

The network was trained on AM1/CIS reference data to predict a **diabatic Hamiltonian**
`W(R)`: a small symmetric matrix over the states S0, S1, S2 whose eigenvalues are the
adiabatic energies and whose off-diagonal elements are the diabatic couplings. Learning `W`
rather than the adiabatic surfaces is convenient for dynamics, because `W` stays a smooth
function of geometry even where the adiabatic states approach each other and their
derivative couplings diverge.

Which quantity the interface reads is selected by `params["label"]`, the name of a node in
the hippynn model. Its shape decides how it is used:

* `(nstates, nstates)` is treated as a diabatic Hamiltonian, couplings included. That is
  what the label `"W"` gives here, and it is what makes nonadiabatic dynamics possible.
* `(nstates,)` is treated as state energies and placed on the diagonal, leaving the states
  uncoupled. Useful for adiabatic dynamics.

The model shipped here sets its label for you, so nothing has to be specified beyond the
dynamics settings themselves.

Nuclear gradients are not read from the network at all: they are obtained by automatic
differentiation of the label with respect to the coordinates, so no derivative labels need
to have been trained or stored.

Before running any dynamics, it is worth evaluating the model once and looking at what it
returns.

<a name="get_atomic_numbers-1"></a>
<a name="get_initial_conditions-1"></a>
<a name="compute_model-1"></a>
The composition of the system and the initial conditions come from the model itself, through
`get_atomic_numbers` and `get_initial_conditions`, and the Hamiltonian is evaluated by
`compute_model`.


In [ ]:
Z = model.get_atomic_numbers()
R, V, masses = model.get_initial_conditions()
print(f"{len(Z)} atoms, {R.shape[0]} initial conditions provided with the model")

# Libra passes coordinates as a MATRIX(ndof, ntraj) in Bohr, ordered x1,y1,z1,x2,y2,z2,...
ndof = 3 * len(Z)
q = MATRIX(ndof, 1)
for i, x in enumerate(R[0].reshape(-1)):
    q.set(i, 0, x)

params = {"model0": 0, "nstates": 3}
obj = model.compute_model(q, params, Py2Cpp_int([0, 0]))

W = np.array([[obj.ham_dia.get(i, j).real for j in range(3)] for i in range(3)])
print("\ndiabatic Hamiltonian W (eV):")
print(np.round(W / units.ev2Ha, 4))

E = np.linalg.eigvalsh(W)
print(f"\nS1-S2 diabatic coupling : {W[1, 2] / units.ev2Ha * 1000.0:.1f} meV")
print(f"adiabatic S1-S2 gap     : {(E[2] - E[1]) / units.ev2Ha * 1000.0:.1f} meV")
print(f"nuclear derivatives     : {len(obj.d1ham_dia)} matrices (= 3 x {len(Z)} atoms)")

Two things to notice.

The S0 row and column of `W` are exactly zero off the diagonal. The ground state is not
coupled to the excited states in the reference data, and the network was built with that
structure imposed, so it holds exactly rather than approximately: dynamics started in the
excited states will never leak into S0.

The S1-S2 coupling is small compared with the gap at this particular geometry. It is
strongly geometry dependent, however, and that dependence is what the dynamics below is
about.


## 3. Choose the Nonadiabatic Dynamics Methodology
<a name="3"></a>[Back to TOC](#TOC)

We run **Ehrenfest** (mean-field) dynamics in the **diabatic** representation:

* `rep_tdse = 0` propagates the electronic wavefunction in the diabatic basis, where the
  coupling is the smooth off-diagonal element of `W` rather than a derivative coupling that
  diverges when the states approach each other;
* `force_method = 2` selects the mean-field force;
* `tsh_method = -1` disables surface hopping, so the propagation is purely mean-field.

The adiabatic populations plotted later are obtained by diagonalizing `W` along the
trajectory, which Libra does internally.


In [ ]:
nsteps = 300               # about 29 fs; increase for a longer trajectory
dt = 4.0                   # a.u. (about 0.097 fs)
prefix = "hippynn_ehrenfest"

dyn_params = {"rep_tdse": 0, "force_method": 2, "tsh_method": -1, "isNBRA": 0,
              "ham_update_method": 1, "ham_transform_method": 1, "time_overlap_method": 1,
              "ntraj": 1, "dt": dt, "nsteps": nsteps, "num_electronic_substeps": 20,
              "prefix": prefix, "prefix2": prefix + "_2",
              "mem_output_level": 3, "hdf5_output_level": -1, "txt_output_level": -1,
              "properties_to_save": ["timestep", "time", "se_pop_adi", "Etot_ave"],
              "progress_frequency": 1.0}

## 4. Choose the initial conditions: Nuclear and Electronic
<a name="4"></a>[Back to TOC](#TOC)

The nuclear initial conditions shipped with the model are geometries and velocities sampled
from a ground-state trajectory at 300 K, which is how a photoexcitation experiment prepares
the molecule. Electronically the system starts in the upper state of the pair, S2: this is
the bright state of this dimer, the one that carries the oscillator strength and is
therefore the one populated by light.


In [ ]:
ic = 0                                        # which initial condition to use

init_nucl = {"init_type": 0, "ndof": ndof,
             "q": list(R[ic].reshape(-1)),
             "p": list((masses[:, None] * V[ic]).reshape(-1)),
             "mass": list(np.repeat(masses, 3))}

init_elec = {"ndia": 3, "nadi": 3, "rep": 1, "init_type": 0,
             "istate": 2,                     # start in S2, the bright state
             "verbosity": 0}

## 5. Running the calculations
<a name="5"></a>[Back to TOC](#TOC)

The call below is the standard Libra dynamics driver. The only thing that distinguishes it
from a run with an analytical model Hamiltonian is that `model.compute_model` evaluates a
neural network.

Each step evaluates the network once and differentiates it a few times, so on a laptop CPU
expect roughly a minute per 100 steps. That is for a single trajectory of a 122-atom
molecule, where the corresponding electronic structure calculation would take orders of
magnitude longer.

<a name="generic_recipe-1"></a>
<a name="compute_model-2"></a>
Here `generic_recipe` drives the dynamics, calling `compute_model` at every step.


In [ ]:
res = tsh_dynamics.generic_recipe(dyn_params, model.compute_model, params,
                                  init_elec, init_nucl, Random())
print("done")

## 6. Plotting the results
<a name="6"></a>[Back to TOC](#TOC)

`se_pop_adi` holds the adiabatic populations of the electronic wavefunction. We also check
the total energy, which is the usual diagnostic that the forces are consistent with the
Hamiltonian. For a learned potential it additionally tells us that the network is smooth
enough to integrate.


In [ ]:
with h5py.File(f"{prefix}/mem_data.hdf", "r") as f:
    pop = np.array(f["se_pop_adi/data"])           # (nsteps, nstates)
    etot = np.array(f["Etot_ave/data"])

time_fs = np.arange(pop.shape[0]) * dt * units.au2fs

plt.figure(figsize=(7, 4))
plt.plot(time_fs, pop[:, 2], label="S2", lw=2)
plt.plot(time_fs, pop[:, 1], label="S1", lw=2)
plt.plot(time_fs, pop[:, 0], label="S0", lw=1, ls="--")
plt.xlabel("time, fs"); plt.ylabel("adiabatic population")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

drift = np.max(np.abs(etot - etot[0])) * units.au2ev * 1000.0
print(f"total energy drift: {drift:.2f} meV")

The population leaves S2 within a few femtoseconds, and the two states then exchange
population back and forth. A single trajectory is noisy, and the oscillations seen in it are
not yet the physical observable.


## 7. The ensemble result
<a name="7"></a>[Back to TOC](#TOC)

The quantity that can be compared with experiment is the **ensemble average** over many
initial conditions. Individual trajectories dephase from one another, and what survives the
averaging is a genuine coherent effect rather than the oscillation of any single trajectory.

The figure below was produced by running the cells above over 300 initial conditions, which
takes about an hour on a GPU and is therefore shown precomputed rather than run live. The
ten initial conditions shipped with the model are enough to reproduce the procedure on a
smaller scale.

It shows the **vibronic beating** of this dimer: after the fast S2 to S1 transfer,
population returns to S2 in recurrences spaced by about 18 fs, decaying over roughly 90 fs.
The period is set by the vibrational modes that carry the molecule back into the strongly
coupled region, and the decay by the dephasing of the ensemble. The same beating is obtained
from full electronic structure calculations on this system, so reproducing it, and in
particular its period and its decay, is a meaningful test that the network learned the
couplings and not merely the energies.


In [ ]:
from IPython.display import Image
Image(filename="beating_ensemble.png")

## 8. Using your own hippynn model
<a name="8"></a>[Back to TOC](#TOC)

The model used above is a thin wrapper: it fills in the network, the label and the atomic
numbers, then calls the general interface. To drive Libra with a model of your own, call
that interface directly.

<a name="hippynn_model-1"></a>
The interface is `libra_py.packages.hippynn.methods.hippynn_model`:

```python
from libra_py.packages.hippynn.methods import hippynn_model

params = {"model0": 0, "nstates": <number of states>,
          "model_path": "<directory with your trained hippynn model>",
          "Z": [<atomic numbers, in the order of the coordinates you pass to Libra>],
          "label": "<name of the node to read>"}

obj = hippynn_model(q, params, full_id)
```

The remaining parameters have sensible defaults and are described in the module's
docstring:

* `params["predictor"]` accepts an already-constructed hippynn `GraphModule` or
  `Predictor` instead of `model_path`, which is useful when your model does not live in a
  standard checkpoint directory or is already in memory.
* `params["energy_units"]` and `params["length_units"]` declare the units your model was
  trained in (`"eV"` and `"Angstrom"` by default); conversion to atomic units happens in
  the interface.
* `params["input_Z"]` and `params["input_R"]` give the names of your model's input nodes,
  if they are not `"Z"` and `"R"`.

Two things are worth checking before running any dynamics.

**Custom node classes have to be importable.** A saved hippynn model records the module
path of every custom node class it was built with, and loading it fails unless those
classes can be imported under the same path. This applies however the model is loaded. It
is also why the model shipped with Libra carries a small sub-package of node definitions
alongside the weights.

**Evaluate once first.** Repeat the single-point call from section 2 with your own model
and inspect the matrix it returns: the units, the symmetry, and whether the couplings have
the magnitude you expect. A units or atom-ordering mistake is far easier to recognize in
one matrix than in a trajectory.

Finally, if your model predicts state energies but no couplings, it can still be used by
pointing `label` at the energies, but the states will be uncoupled and the dynamics
adiabatic. Nonadiabatic dynamics needs a model that predicts the coupled Hamiltonian, as
here.
